# 05 - Comparação de modelos: FP-Growth × Item2Vec no mesmo hold-out (Fase 2 · ML)
**Squad 2 — Negócio: ML em Streaming | P2 · Recomendação de produtos no carrinho**
**Responsável:** Zaiden Emiliano Segundo Seleme · **Branch:** `feat/squad2-fase2-ml-zaiden`
**Modelo A (regras de associação):** FP-Growth v2 do Lucas Sousa · **Modelo B (embeddings):** Item2Vec v1 (notebook 03)

---

### Objetivo
Comparar os dois modelos do P2 **na mesma prova**: quais entradas cada um usa, em que tipo de dado cada um funciona melhor e qual acerta mais. É a base do one page de comparação.

### Por que este notebook existe
Os números de cada um não eram comparáveis:

| | FP-Growth v2 (Lucas) | Item2Vec v1 (notebook 03) |
|---|---|---|
| Split | aleatório 80/20 | temporal (as 20% cestas mais recentes) |
| Cestas avaliadas | 29 (só o período "noite", o único que gerou regras) | 336 (todas as cestas de teste) |
| Hit@5 | 6,9% (2 de 29, IC 95% de 1,9% a 22%) | 4,17% (14 de 336, IC 95% de 2,5% a 6,9%) |

Aqui os dois rodam com o **mesmo treino** (`item2vec_cestas_treino`) e o **mesmo teste** (`item2vec_holdout`, 336 cestas, 1 item removido por cesta). Todas as cestas de teste entram na métrica, inclusive aquelas em que o modelo não tem regra ou vetor (aí vale o fallback).

### Regras da comparação
* **O FP-Growth é reimplementado com a lógica da v2 do Lucas**: regras por período do dia, `min_support` 0,01, `min_confidence` 0,2, filtro `lift > 1,2`, pisos de volume (30 cestas, 2 ocorrências), cascata SKU → categoria → popularidade. Usamos `mlxtend` porque o Free Edition não tem MLlib (`pyspark.ml.fpm`). O algoritmo e os limiares são os mesmos.
* **Nenhum parâmetro é escolhido olhando o teste.** O piso de ocorrências do FP-Growth calibrado é escolhido numa **validação dentro do treino**. O Item2Vec usa os parâmetros fixados no one page (notebook 03).
* **Incerteza sempre junto da métrica:** intervalo de confiança de Wilson (95%) e teste de McNemar, que compara os dois modelos cesta a cesta.

### Entradas e saídas
| Lê (Gold, notebook 02) | Grava (Gold) |
|---|---|
| `item2vec_cestas_treino`, `item2vec_holdout`, `item2vec_skus` | `p2_comparacao_avaliacao` (métricas por modelo, em append) e `p2_comparacao_recomendacoes` (top-5 de cada carrinho, por modelo) |

In [ ]:
%pip install gensim==4.4.0 mlxtend==0.25.0 python-dotenv==1.0.1 --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Ambiente, caminhos e parâmetros

In [ ]:
import os
import math
import hashlib
from collections import defaultdict
from datetime import datetime
from itertools import combinations
from dotenv import load_dotenv, find_dotenv
import numpy as np
import pandas as pd
import gensim
import mlxtend
from gensim.models import Word2Vec
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed/.env",
        "/Workspace/Repos/estagio_ed/.env",
    ]
    dotenv_path = next((c for c in candidatos if os.path.isfile(c)), None)
load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
assert storage_account and client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

fqdn = f"{storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

GOLD = f"abfss://squad2@{fqdn}/fase2_ml/gold"
TABELAS = {nome: f"{GOLD}/item2vec_{nome}" for nome in ["skus", "cestas_treino", "holdout"]}
DESTINOS = {nome: f"{GOLD}/p2_comparacao_{nome}" for nome in ["avaliacao", "recomendacoes"]}

def ler_delta(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

def gravar_delta(df, destino, modo="overwrite"):
    w = df.write.format("delta").options(**adls_options).mode(modo)
    (w.option("overwriteSchema", "true") if modo == "overwrite" else w.option("mergeSchema", "true")).save(destino)

print(f"gensim {gensim.__version__} | mlxtend {mlxtend.__version__}")

In [ ]:
VERSAO = "v1_comparacao"
TOP_K = 5
FUSO = "America/Sao_Paulo"

# FP-Growth v2 do Lucas: limiares copiados do notebook dele, sem alteração
LUCAS = {"min_support": 0.01, "min_confidence": 0.2, "min_lift": 1.2,
         "min_cestas_treino": 30, "min_count_minimo": 2}
PISOS_CANDIDATOS = [2, 3, 4, 5]   # ocorrências mínimas testadas na validação do FP-Growth calibrado
MAX_LEN = 3                       # itemsets de até 3 SKUs: com piso baixo, sem limite a mineração explode

# Item2Vec v1: os mesmos parâmetros do notebook 03 (fixados no one page)
PARAMS_I2V = {"vector_size": 32, "window": 24, "min_count": 1, "sg": 1,
              "negative": 5, "epochs": 50, "seed": 42, "workers": 1}
SEEDS_VARIACAO = list(range(1, 11))   # para medir quanto o Item2Vec varia só pela seed
FRACAO_VALIDACAO = 0.2                # últimas 20% cestas do treino viram validação

## 1. Leitura do treino e do teste
As mesmas tabelas do notebook 03, em pandas (são pequenas). O período do dia (madrugada/manhã/tarde/noite) é derivado da hora **no fuso de Brasília**, com os mesmos cortes do Lucas (6h, 12h e 18h).

> O notebook do Lucas não define `spark.sql.session.timeZone`, então lá a hora sai em UTC e os períodos ficam deslocados 3h (a "tarde" UTC é 9h–15h em Brasília). Aqui seguimos o fuso oficial da squad.

In [ ]:
skus = ler_delta(TABELAS["skus"]).toPandas()
treino = ler_delta(TABELAS["cestas_treino"]).orderBy("id_pedido").toPandas()
holdout = ler_delta(TABELAS["holdout"]).orderBy("id_pedido").toPandas()

In [ ]:
def hora_local(serie):
    """Hora no fuso de Brasília. O toPandas do Databricks já entrega no fuso da sessão (sem tz);
    fora dele o timestamp chega com tz e é convertido."""
    s = pd.to_datetime(serie)
    return (s.dt.tz_convert(FUSO) if s.dt.tz is not None else s).dt.hour

def periodo_do_dia(hora):
    return "madrugada" if hora < 6 else "manha" if hora < 12 else "tarde" if hora < 18 else "noite"

treino["periodo"] = hora_local(treino["dt_pedido"]).map(periodo_do_dia)
holdout["periodo"] = hora_local(holdout["dt_pedido"]).map(periodo_do_dia)
cestas_treino = [list(s) for s in treino["skus"]]
subcategoria_de = dict(zip(skus["sku"], skus["subcategoria"]))

print(f"Treino: {len(treino):,} cestas | teste: {len(holdout):,} cestas | catálogo: {len(skus):,} SKUs")
display(pd.DataFrame({"treino": treino["periodo"].value_counts(),
                      "teste": holdout["periodo"].value_counts()}).fillna(0).astype(int)
          .rename_axis("periodo").reset_index())

## 2. Diagnóstico: por que a v2 do Lucas não gera regras de SKU nesta base
O `min_support` é uma **fração** das cestas: o mínimo de ocorrências para um par virar "frequente" é `ceil(min_support × nº de cestas)`. Se nenhum par de SKUs aparece tantas vezes, não sai nenhuma regra de SKU, e a recomendação cai inteira no fallback.

In [ ]:
pares = pd.Series([p for c in cestas_treino for p in combinations(sorted(set(c)), 2)]).value_counts()
mais_frequente = pd.Series([s for c in cestas_treino for s in c]).value_counts()

print(f"SKU mais vendido: em {mais_frequente.iloc[0]} de {len(cestas_treino):,} cestas "
      f"({mais_frequente.iloc[0] / len(cestas_treino):.1%})")
print(f"Pares de SKUs distintos: {len(pares):,} | o mais frequente aparece em {pares.iloc[0]} cestas")
display(pares.value_counts().sort_index().rename_axis("cestas_com_o_par").rename("pares_distintos").reset_index())

for periodo, grupo in treino.groupby("periodo"):
    n = len(grupo)
    print(f"Período {periodo:>9}: {n:4d} cestas de treino -> mínimo de ocorrências = "
          f"ceil({LUCAS['min_support']} × {n}) = {math.ceil(LUCAS['min_support'] * n)}")

## 3. Peças comuns: mineração, regras e popularidade
* `minerar`: FP-Growth + regras de associação com os filtros de confiança e lift, e os dois pisos de volume do Lucas (`min_cestas_treino` e `min_count_minimo`). Abaixo dos pisos, não treina.
* `Regras`: índice antecedente → regras, para achar rápido as regras cujo antecedente está **contido** no carrinho (o mesmo casamento que o Lucas faz em DataFrame).
* `consequentes`: junta os consequentes das regras que casaram. Ordem **alfabética** (o que o `array_sort` do Lucas faz) ou por **confiança** (depois lift).
* `preparar_contexto`: popularidade geral e por subcategoria, mais as regras de categoria, calculadas só com o treino recebido. Reusado na validação.

In [ ]:
COLUNAS_REGRA = ["antecedents", "consequents", "support", "confidence", "lift"]

def minerar(cestas, min_support, min_confidence=LUCAS["min_confidence"], min_lift=LUCAS["min_lift"],
            max_len=None, min_cestas=LUCAS["min_cestas_treino"], min_count_minimo=LUCAS["min_count_minimo"]):
    n = len(cestas)
    if n < min_cestas or math.ceil(min_support * n) < min_count_minimo:
        return pd.DataFrame(columns=COLUNAS_REGRA)
    te = TransactionEncoder()
    X = pd.DataFrame(te.fit(cestas).transform(cestas), columns=te.columns_)
    frequentes = fpgrowth(X, min_support=min_support, use_colnames=True, max_len=max_len)
    if frequentes.empty or frequentes["itemsets"].map(len).max() < 2:
        return pd.DataFrame(columns=COLUNAS_REGRA)
    regras = association_rules(frequentes, metric="confidence", min_threshold=min_confidence, num_itemsets=n)
    return regras[regras["lift"] > min_lift][COLUNAS_REGRA].reset_index(drop=True)


class Regras:
    def __init__(self, df):
        self.df = df
        self.por_antecedente = defaultdict(list)
        for ant, cons, conf, lift in zip(df["antecedents"], df["consequents"], df["confidence"], df["lift"]):
            self.por_antecedente[frozenset(ant)].append((tuple(sorted(cons)), conf, lift))
        self.maior_antecedente = max((len(a) for a in self.por_antecedente), default=0)

    def aplicaveis(self, itens):
        """Regras cujo antecedente está contido em `itens`."""
        itens = set(itens)
        n_combinacoes = sum(math.comb(len(itens), k) for k in range(1, self.maior_antecedente + 1))
        if n_combinacoes <= len(self.por_antecedente):
            chaves = (frozenset(c) for k in range(1, self.maior_antecedente + 1)
                      for c in combinations(sorted(itens), k))
            chaves = [c for c in chaves if c in self.por_antecedente]
        else:
            chaves = [a for a in self.por_antecedente if a <= itens]
        return [r for a in chaves for r in self.por_antecedente[a]]


def consequentes(regras, itens, ordem):
    """ordem='alfabetica' (como o array_sort do Lucas) ou 'confianca' (maior confiança, depois lift)."""
    melhor = {}
    for cons, conf, lift in regras.aplicaveis(itens):
        for sku in cons:
            if sku not in melhor or (conf, lift) > melhor[sku]:
                melhor[sku] = (conf, lift)
    if ordem == "alfabetica":
        return sorted(melhor)
    return [s for s, _ in sorted(melhor.items(), key=lambda kv: (-kv[1][0], -kv[1][1], kv[0]))]


def completar(recs, carrinho, candidatos):
    """Acrescenta candidatos que não estão no carrinho nem já recomendados, até TOP_K."""
    for sku in candidatos:
        if len(recs) >= TOP_K:
            break
        if sku not in carrinho and sku not in recs:
            recs.append(sku)
    return recs


def preparar_contexto(cestas, freq=None):
    """Popularidade (geral e por subcategoria) e regras de categoria, só com o treino recebido."""
    if freq is None:
        freq = pd.Series([s for c in cestas for s in c]).value_counts()
    populares = sorted(freq[freq > 0].index, key=lambda s: (-freq[s], s))
    por_sub = defaultdict(list)
    for sku in populares:
        por_sub[subcategoria_de.get(sku)].append(sku)
    cestas_categoria = [sorted({subcategoria_de[s] for s in c}) for c in cestas]
    cestas_categoria = [c for c in cestas_categoria if len(c) >= 2]
    return {"mais_vendidos": populares, "mais_vendidos_sub": dict(por_sub),
            "regras_cat": Regras(minerar(cestas_categoria, LUCAS["min_support"]))}


# contexto do treino completo: a popularidade vem de item2vec_skus, como no notebook 03
ctx = preparar_contexto(cestas_treino, freq=skus.set_index("sku")["freq_treino"])
print(f"Regras de categoria (subcategoria, min_support {LUCAS['min_support']}): {len(ctx['regras_cat'].df):,}")

## 4. Os recomendadores
Todos recebem `(carrinho, período, contexto)` e devolvem `(top-5, n_modelo)`. `n_modelo` é quantas das 5 sugestões vieram do **próprio modelo** (regras de SKU ou vetores), e não do fallback. Isso mostra quanto do acerto é mérito do modelo.

| Modelo | Camada principal | Fallback |
|---|---|---|
| **Top-5 mais vendidos** | — | mais vendidos do treino |
| **FP-Growth v2 (Lucas)** | regras de SKU do período do pedido, ordem alfabética | regras de categoria → SKU mais vendido de cada categoria prevista → 15 mais vendidos |
| **FP-Growth calibrado** | regras de SKU globais, piso escolhido na validação, ordem por confiança | regras de categoria → mais vendidos das subcategorias do carrinho → mais vendidos |
| **Item2Vec v1** | cosseno com a média dos vetores do carrinho | mais vendidos das subcategorias do carrinho → mais vendidos |

O FP-Growth calibrado muda três coisas em relação à v2, e a seção 8 mede o efeito de cada uma:
1. **Sem segmentar por período.** O treino é 73% manhã e o teste 94% tarde: regras por período treinam num horário e testam em outro. A EDA (notebook 01) também mostrou |r| ≤ 0,20 entre hora e conteúdo da cesta.
2. **Piso de ocorrências no lugar do `min_support` fixo.** O próprio Lucas definiu `min_count_minimo = 2` como piso; aqui o piso é escolhido na validação.
3. **Ordem por confiança** em vez de alfabética. Com a alfabética, entre 20 consequentes o top-5 é quem começa com "a".

In [ ]:
def rec_populares(carrinho, periodo, ctx):
    return completar([], set(carrinho), ctx["mais_vendidos"]), 0


def rec_fpgrowth_v2(carrinho, periodo, ctx):
    """Cascata do Lucas: SKU do período -> categoria (SKU mais vendido de cada categoria prevista) -> populares."""
    no_carrinho = set(carrinho)
    regras_periodo = ctx["regras_periodo"].get(periodo)
    rec_sku = consequentes(regras_periodo, carrinho, "alfabetica")[:TOP_K] if regras_periodo else []
    categorias = consequentes(ctx["regras_cat"], {subcategoria_de[s] for s in carrinho}, "alfabetica")[:TOP_K]
    rec_cat = sorted({ctx["mais_vendidos_sub"][c][0] for c in categorias if c in ctx["mais_vendidos_sub"]})
    recs = completar([], no_carrinho, rec_sku)
    n_modelo = len(recs)
    recs = completar(recs, no_carrinho, rec_cat)
    return completar(recs, no_carrinho, ctx["mais_vendidos"][:TOP_K * 3]), n_modelo


def rec_fpgrowth_calibrado(carrinho, periodo, ctx):
    no_carrinho = set(carrinho)
    recs = completar([], no_carrinho, consequentes(ctx["regras_sku"], carrinho, "confianca"))
    n_modelo = len(recs)
    for categoria in consequentes(ctx["regras_cat"], {subcategoria_de[s] for s in carrinho}, "confianca"):
        recs = completar(recs, no_carrinho, ctx["mais_vendidos_sub"].get(categoria, [])[:1])
    for sub in dict.fromkeys(subcategoria_de.get(s) for s in carrinho):
        recs = completar(recs, no_carrinho, ctx["mais_vendidos_sub"].get(sub, []))
    return completar(recs, no_carrinho, ctx["mais_vendidos"]), n_modelo


def rec_item2vec(carrinho, periodo, ctx):
    """Igual ao notebook 03: cosseno com a média do carrinho + fallback por subcategoria."""
    wv = ctx["w2v"].wv
    no_carrinho = set(carrinho)
    conhecidos = [s for s in carrinho if s in wv]
    recs = []
    if conhecidos:
        perfil = np.mean([wv[s] for s in conhecidos], axis=0)
        for sku, _ in wv.similar_by_vector(perfil, topn=TOP_K + len(conhecidos)):
            if sku not in no_carrinho:
                recs.append(sku)
            if len(recs) == TOP_K:
                break
    n_modelo = len(recs)
    for sub in dict.fromkeys(subcategoria_de.get(s) for s in carrinho):
        recs = completar(recs, no_carrinho, ctx["mais_vendidos_sub"].get(sub, []))
    return completar(recs, no_carrinho, ctx["mais_vendidos"]), n_modelo

## 5. Avaliação e incerteza
* **Hit@5**, **Precision@5** (= Hit@5 ÷ 5, porque há 1 item certo por cesta), **cobertura do catálogo** e **acerto de subcategoria**: as mesmas definições do notebook 03.
* **IC de Wilson (95%):** a faixa em que o Hit@5 verdadeiro deve estar. Com poucas cestas a faixa é larga, e dois modelos com faixas sobrepostas podem estar empatados.
* **McNemar (exato):** como os dois modelos respondem às **mesmas** cestas, a comparação justa olha só as cestas em que um acertou e o outro errou. Se essas discordâncias fossem sorteio 50/50, o p-valor diz quão raro seria um placar tão desequilibrado. p < 0,05 indica diferença real.

In [ ]:
def wilson(acertos, n, z=1.959963984540054):
    if n == 0:
        return 0.0, 0.0
    p = acertos / n
    d = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / d
    margem = z / d * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, centro - margem), min(1.0, centro + margem)


def mcnemar_exato(acertos_a, acertos_b):
    """Teste binomial exato bicaudal sobre as cestas discordantes."""
    so_a = int(np.sum(acertos_a & ~acertos_b))
    so_b = int(np.sum(~acertos_a & acertos_b))
    n = so_a + so_b
    if n == 0:
        return so_a, so_b, 1.0
    cauda = sum(math.comb(n, i) for i in range(min(so_a, so_b) + 1)) / 2 ** n
    return so_a, so_b, min(1.0, 2 * cauda)


def avaliar(nome, recomendar, base, ctx):
    linhas, recomendados = [], set()
    for cesta in base.itertuples(index=False):
        carrinho = list(cesta.carrinho)
        recs, n_modelo = recomendar(carrinho, cesta.periodo, ctx)
        assert len(recs) == TOP_K and not set(recs) & set(carrinho), f"{nome}: top-5 inválido {recs}"
        recomendados.update(recs)
        linhas.append({"id_pedido": int(cesta.id_pedido), "modelo": nome, "recomendacoes": recs,
                       "item_removido": cesta.item_removido, "acerto": cesta.item_removido in recs,
                       "acerto_do_modelo": cesta.item_removido in recs[:n_modelo],
                       "modelo_respondeu": n_modelo > 0,
                       "acerto_subcategoria": subcategoria_de.get(cesta.item_removido)
                                              in {subcategoria_de.get(r) for r in recs}})
    df = pd.DataFrame(linhas)
    n, k = len(df), int(df["acerto"].sum())
    ic_inf, ic_sup = wilson(k, n)
    metricas = {"modelo": nome, "cestas": n, "acertos": k, "hit_at_5": k / n,
                "ic95_inf": ic_inf, "ic95_sup": ic_sup, "precision_at_5": k / n / TOP_K,
                "cobertura_catalogo": len(recomendados) / len(skus), "skus_recomendados": len(recomendados),
                "acerto_subcategoria": df["acerto_subcategoria"].mean(),
                "modelo_respondeu": df["modelo_respondeu"].mean(),
                "acertos_do_modelo": int(df["acerto_do_modelo"].sum())}
    return metricas, df


def tabela(metricas):
    t = pd.DataFrame(metricas)
    for c in ["hit_at_5", "precision_at_5", "cobertura_catalogo", "acerto_subcategoria", "modelo_respondeu"]:
        t[c] = (100 * t[c]).round(2).astype(str) + "%"
    t["ic95"] = [f"{100 * a:.1f}% a {100 * b:.1f}%" for a, b in zip(t.pop("ic95_inf"), t.pop("ic95_sup"))]
    return t

## 6. Escolha do piso do FP-Growth calibrado (validação dentro do treino)
As **últimas 20% cestas do treino** (por instante do pedido) viram validação, com o mesmo protocolo do hold-out: 1 item removido por cesta. O item é escolhido pelo menor `md5(id_pedido|sku)`, uma regra determinística, como o `xxhash64` do notebook 02. Popularidade, regras de categoria e regras de SKU são recalculadas **só com os 80% iniciais**. Vence o piso com maior Hit@5 na validação. O hold-out não participa da escolha.

O Item2Vec também é medido na validação, com os parâmetros do notebook 03 e 10 seeds, só como referência.

> A validação vem das tabelas da Gold, que já passaram pelo filtro de vocabulário do notebook 02. Por isso ela não tem itens "nunca vistos" e os números saem **maiores** que no hold-out, onde 29% dos itens removidos estão fora do vocabulário.

In [ ]:
def item_validacao(id_pedido, cesta):
    return min(cesta, key=lambda s: hashlib.md5(f"{id_pedido}|{s}".encode()).hexdigest())

ordenado = treino.sort_values(["dt_pedido", "id_pedido"]).reset_index(drop=True)
corte = int(len(ordenado) * (1 - FRACAO_VALIDACAO))
sub_treino, sub_val = ordenado.iloc[:corte], ordenado.iloc[corte:]
cestas_sub_treino = [list(s) for s in sub_treino["skus"]]

validacao = pd.DataFrame({"id_pedido": sub_val["id_pedido"].values, "periodo": sub_val["periodo"].values,
                          "item_removido": [item_validacao(i, c) for i, c in zip(sub_val["id_pedido"], sub_val["skus"])]})
validacao["carrinho"] = [sorted(set(c) - {r}) for c, r in zip(sub_val["skus"], validacao["item_removido"])]
ctx_val = preparar_contexto(cestas_sub_treino)
print(f"Validação: {len(cestas_sub_treino):,} cestas para treinar | {len(validacao):,} para validar "
      f"(de {sub_val['dt_pedido'].min()} a {sub_val['dt_pedido'].max()})")

escolha = []
for piso in PISOS_CANDIDATOS:
    ctx_val["regras_sku"] = Regras(minerar(cestas_sub_treino, piso / len(cestas_sub_treino), max_len=MAX_LEN))
    m, _ = avaliar(f"fpgrowth_piso_{piso}", rec_fpgrowth_calibrado, validacao, ctx_val)
    escolha.append({**m, "piso": piso, "regras": len(ctx_val["regras_sku"].df)})
escolha = pd.DataFrame(escolha)
PISO = int(escolha.sort_values(["hit_at_5", "piso"], ascending=[False, True]).iloc[0]["piso"])

hits_val_i2v = []
for seed in SEEDS_VARIACAO:
    ctx_val["w2v"] = Word2Vec(sentences=cestas_sub_treino, **{**PARAMS_I2V, "seed": seed})
    hits_val_i2v.append(avaliar("item2vec", rec_item2vec, validacao, ctx_val)[0]["hit_at_5"])

display(tabela(escolha[["modelo", "piso", "regras", "cestas", "acertos", "hit_at_5", "ic95_inf", "ic95_sup",
                        "precision_at_5", "cobertura_catalogo", "acerto_subcategoria", "modelo_respondeu"]].to_dict("records")))
print(f"Piso escolhido para o FP-Growth calibrado: {PISO} ocorrências")
print(f"Item2Vec v1 na validação (10 seeds): média {np.mean(hits_val_i2v):.2%} "
      f"(de {min(hits_val_i2v):.2%} a {max(hits_val_i2v):.2%})")

## 7. Treino dos modelos no treino completo
* **FP-Growth v2:** um modelo por período do dia, com os limiares e os pisos do Lucas.
* **FP-Growth calibrado:** regras globais com o piso escolhido na seção 6.
* **Item2Vec v1:** seed 42, como no notebook 03 (deve reproduzir os 4,17%), mais 10 seeds para medir a variação.

In [ ]:
ctx["regras_periodo"] = {}
for periodo, grupo in treino.groupby("periodo"):
    cestas_periodo = [list(s) for s in grupo["skus"]]
    ctx["regras_periodo"][periodo] = Regras(minerar(cestas_periodo, LUCAS["min_support"]))
    print(f"FP-Growth v2 | período {periodo:>9}: {len(cestas_periodo):4d} cestas -> "
          f"{len(ctx['regras_periodo'][periodo].df):,} regras de SKU")

ctx["regras_sku"] = Regras(minerar(cestas_treino, PISO / len(cestas_treino), max_len=MAX_LEN))
print(f"FP-Growth calibrado | piso {PISO}: {len(ctx['regras_sku'].df):,} regras de SKU")

inicio = datetime.now()
ctx["w2v"] = Word2Vec(sentences=cestas_treino, **PARAMS_I2V)
print(f"Item2Vec v1 | {len(ctx['w2v'].wv):,} vetores em {(datetime.now() - inicio).total_seconds():.1f}s")

## 8. Resultado no hold-out (336 cestas)

In [ ]:
MODELOS = {"top5_mais_vendidos": rec_populares,
           "fpgrowth_v2_lucas": rec_fpgrowth_v2,
           "fpgrowth_calibrado": rec_fpgrowth_calibrado,
           "item2vec_v1": rec_item2vec}
resultados = {nome: avaliar(nome, f, holdout, ctx) for nome, f in MODELOS.items()}
metricas = [m for m, _ in resultados.values()]
display(tabela(metricas)[["modelo", "cestas", "acertos", "hit_at_5", "ic95", "precision_at_5", "cobertura_catalogo",
                          "skus_recomendados", "acerto_subcategoria", "modelo_respondeu", "acertos_do_modelo"]])

acertos = {nome: df["acerto"].to_numpy() for nome, (_, df) in resultados.items()}
duelos = []
for a, b in [("item2vec_v1", "fpgrowth_calibrado"), ("item2vec_v1", "fpgrowth_v2_lucas"),
             ("fpgrowth_calibrado", "top5_mais_vendidos"), ("item2vec_v1", "top5_mais_vendidos")]:
    so_a, so_b, p = mcnemar_exato(acertos[a], acertos[b])
    duelos.append({"A": a, "B": b, "só A acertou": so_a, "só B acertou": so_b, "p (McNemar)": round(p, 4),
                   "diferença real?": "sim" if p < 0.05 else "não (empate estatístico)"})
display(pd.DataFrame(duelos))

### Variação do Item2Vec entre seeds
O FP-Growth é determinístico: mesmos dados, mesmas regras. O Item2Vec depende da inicialização aleatória dos vetores. Uma execução só pode estar acima ou abaixo da média por sorte.

In [ ]:
hits_seeds = []
for seed in SEEDS_VARIACAO:
    ctx_seed = {**ctx, "w2v": Word2Vec(sentences=cestas_treino, **{**PARAMS_I2V, "seed": seed})}
    hits_seeds.append(avaliar("item2vec", rec_item2vec, holdout, ctx_seed)[0]["acertos"])
print(f"Item2Vec v1 em 10 seeds: acertos {hits_seeds} -> média {np.mean(hits_seeds) / len(holdout):.2%} "
      f"(de {min(hits_seeds) / len(holdout):.2%} a {max(hits_seeds) / len(holdout):.2%})")

### De onde vem o acerto: cestas que repetem o treino
Uma parte das cestas do stream é **cópia exata** de outra: mesmo conjunto de SKUs, pedido diferente. Aqui marcamos as cestas de teste cuja cesta completa (carrinho + item removido) está contida numa cesta de treino. Separar o Hit@5 entre "repete o treino" e "cesta nova" mostra se o modelo **generaliza** ou só **lembra** combinações já vistas.

In [ ]:
conjuntos_treino = [frozenset(s) for s in cestas_treino]
repeticoes = pd.Series(conjuntos_treino).value_counts()
copias = int(repeticoes[repeticoes > 1].sum())
print(f"Cestas de treino que repetem exatamente outra cesta: {copias:,} de {len(conjuntos_treino):,} "
      f"({copias / len(conjuntos_treino):.0%}), em {int((repeticoes > 1).sum())} conjuntos distintos")

holdout["repete_treino"] = [any(frozenset([*c, r]) <= s for s in conjuntos_treino)
                            for c, r in zip(holdout["carrinho"], holdout["item_removido"])]
repete = holdout.set_index("id_pedido")["repete_treino"]
origem = []
for nome, (_, df) in resultados.items():
    df = df.assign(repete_treino=df["id_pedido"].map(repete))
    for valor, rotulo_grupo in [(True, "repete o treino"), (False, "cesta nova")]:
        g = df[df["repete_treino"] == valor]
        origem.append({"modelo": nome, "grupo": rotulo_grupo, "cestas": len(g), "acertos": int(g["acerto"].sum()),
                       "hit_at_5": f"{g['acerto'].mean():.2%}"})
display(pd.DataFrame(origem))

### O que levou o FP-Growth de 1,5% para ~6%
Cada linha liga uma mudança por vez, a partir da v2 do Lucas. É a resposta para "o que mudar no pipeline dele".

O piso também é testado direto no hold-out, só como **diagnóstico de robustez**: a escolha já foi feita na seção 6. Se o acerto despenca com o piso 3, os acertos dependem de regras vistas em apenas 2 cestas.

In [ ]:
regras_seg_piso = {}
for periodo, grupo in treino.groupby("periodo"):
    cestas_periodo = [list(s) for s in grupo["skus"]]
    regras_seg_piso[periodo] = Regras(minerar(cestas_periodo, PISO / max(len(cestas_periodo), 1), max_len=MAX_LEN))

def cascata_lucas(rec_sku, carrinho, ctx):
    categorias = consequentes(ctx["regras_cat"], {subcategoria_de[s] for s in carrinho}, "alfabetica")[:TOP_K]
    rec_cat = sorted({ctx["mais_vendidos_sub"][c][0] for c in categorias if c in ctx["mais_vendidos_sub"]})
    no_carrinho = set(carrinho)
    recs = completar([], no_carrinho, rec_sku[:TOP_K])
    n_modelo = len(recs)
    recs = completar(recs, no_carrinho, rec_cat)
    return completar(recs, no_carrinho, ctx["mais_vendidos"][:TOP_K * 3]), n_modelo

etapas = {
    "1. v2 do Lucas (min_support 0,01, por período, alfabética)": rec_fpgrowth_v2,
    f"2. + piso de {PISO} ocorrências (ainda por período)":
        lambda c, p, x: cascata_lucas(consequentes(regras_seg_piso[p], c, "alfabetica")
                                      if regras_seg_piso.get(p) and len(regras_seg_piso[p].df) else [], c, x),
    "3. + sem segmentar por período":
        lambda c, p, x: cascata_lucas(consequentes(x["regras_sku"], c, "alfabetica"), c, x),
    "4. + ordem por confiança": lambda c, p, x: cascata_lucas(consequentes(x["regras_sku"], c, "confianca"), c, x),
    "5. + fallback por subcategoria do carrinho (= calibrado)": rec_fpgrowth_calibrado,
}
ablacao = [avaliar(nome, f, holdout, ctx)[0] for nome, f in etapas.items()]
display(tabela(ablacao)[["modelo", "acertos", "hit_at_5", "ic95", "cobertura_catalogo", "acertos_do_modelo"]])

robustez = []
for piso in PISOS_CANDIDATOS:
    ctx_piso = {**ctx, "regras_sku": Regras(minerar(cestas_treino, piso / len(cestas_treino), max_len=MAX_LEN))}
    m, _ = avaliar(f"fpgrowth_calibrado_piso_{piso}", rec_fpgrowth_calibrado, holdout, ctx_piso)
    robustez.append({**m, "regras": len(ctx_piso["regras_sku"].df)})
display(tabela(robustez)[["modelo", "regras", "acertos", "hit_at_5", "ic95", "cobertura_catalogo"]])

## 9. Exemplos: o mesmo carrinho nos dois modelos
Cestas em que só um dos modelos acertou, com a subcategoria de cada SKU. Para o acerto do FP-Growth, aparece também a regra que gerou a sugestão e **em quantas cestas de treino** ela se apoia.

In [ ]:
df_fp = resultados["fpgrowth_calibrado"][1].set_index("id_pedido")
df_i2v = resultados["item2vec_v1"][1].set_index("id_pedido")
carrinhos = holdout.set_index("id_pedido")["carrinho"]

def rotulo(sku):
    return f"{sku}  [{subcategoria_de.get(sku)}]"

def mostrar(id_pedido, titulo):
    print(f"=== {titulo} | pedido {id_pedido}")
    print("Carrinho:", *[rotulo(s) for s in carrinhos[id_pedido]], sep="\n   ")
    print("Item removido:", rotulo(df_fp.loc[id_pedido, "item_removido"]))
    for nome, df in [("FP-Growth calibrado", df_fp), ("Item2Vec v1", df_i2v)]:
        marca = "ACERTOU" if df.loc[id_pedido, "acerto"] else "errou"
        print(f"{nome} ({marca}):", *[rotulo(s) for s in df.loc[id_pedido, "recomendacoes"]], sep="\n   ")
    print()

so_fp = df_fp.index[df_fp["acerto"] & ~df_i2v["acerto"]]
so_i2v = df_fp.index[~df_fp["acerto"] & df_i2v["acerto"]]
if len(so_fp):
    mostrar(so_fp[0], "Só o FP-Growth acertou")
if len(so_i2v):
    mostrar(so_i2v[0], "Só o Item2Vec acertou")

# a regra por trás dos acertos do FP-Growth: em quantas cestas de treino ela aparece?
suporte_acertos = []
for id_pedido in df_fp.index[df_fp["acerto_do_modelo"]]:
    alvo, carrinho = df_fp.loc[id_pedido, "item_removido"], set(carrinhos[id_pedido])
    regras = ctx["regras_sku"].df
    usadas = regras[regras["consequents"].map(lambda c: alvo in c) & regras["antecedents"].map(lambda a: a <= carrinho)]
    melhor = usadas.sort_values(["confidence", "lift"], ascending=False).iloc[0]
    suporte_acertos.append({"id_pedido": id_pedido, "antecedente": sorted(melhor["antecedents"]),
                            "consequente": sorted(melhor["consequents"]), "confianca": round(melhor["confidence"], 2),
                            "lift": round(melhor["lift"], 1),
                            "cestas_de_treino": round(melhor["support"] * len(cestas_treino))})
suporte_acertos = pd.DataFrame(suporte_acertos)
display(suporte_acertos)
print("Cestas de treino que sustentam a regra de cada acerto:",
      suporte_acertos["cestas_de_treino"].value_counts().sort_index().to_dict())

## 10. Gravação na Gold
* `p2_comparacao_avaliacao`: uma linha por modelo, com o IC e os parâmetros, em **append** (a próxima rodada entra com outra `versao`, por exemplo quando o Item2Vec v2 do notebook de evolução estiver pronto).
* `p2_comparacao_recomendacoes`: o top-5 de cada carrinho de teste por modelo (overwrite), para auditar acertos e erros.

In [ ]:
avaliado_em = datetime.now()
params = {"top5_mais_vendidos": "{}",
          "fpgrowth_v2_lucas": str(LUCAS),
          "fpgrowth_calibrado": str({**LUCAS, "min_count": PISO, "max_len": MAX_LEN, "segmentado": False, "ordem": "confianca"}),
          "item2vec_v1": str(PARAMS_I2V)}
aval = pd.DataFrame(metricas)
aval["params"] = aval["modelo"].map(params)
aval["versao"] = VERSAO
aval["avaliado_em"] = avaliado_em
gravar_delta(spark.createDataFrame(aval), DESTINOS["avaliacao"], modo="append")

recs = pd.concat([df for _, df in resultados.values()], ignore_index=True)
recs = recs[["id_pedido", "modelo", "recomendacoes", "item_removido", "acerto", "acerto_do_modelo"]]
recs["versao"] = VERSAO
gravar_delta(spark.createDataFrame(recs), DESTINOS["recomendacoes"])

print(f"Avaliações acumuladas: {ler_delta(DESTINOS['avaliacao']).count():,} | "
      f"recomendações: {ler_delta(DESTINOS['recomendacoes']).count():,} linhas")

## 11. Resultado

> Execução local em 23/11/2026, com as tabelas da Gold lidas do ADLS. É o mesmo código; só a leitura e a gravação trocam o Spark por parquet. O Item2Vec v1 reproduziu exatamente os 14 acertos do notebook 03. **Falta confirmar no Databricks.**

| Modelo | Hit@5 | IC 95% | Precision@5 | Cobertura | Acerto de subcategoria |
|---|---|---|---|---|---|
| Top-5 mais vendidos | 1,49% | 0,6% a 3,4% | 0,30% | 0,18% (7 SKUs) | 16,4% |
| FP-Growth v2 (parâmetros do Lucas) | 1,49% | 0,6% a 3,4% | 0,30% | 1,0% (39 SKUs) | 36,6% |
| FP-Growth calibrado (piso 2, escolhido na validação) | **5,95%** | 3,9% a 9,0% | 1,19% | 11,6% (443 SKUs) | **39,0%** |
| Item2Vec v1 (seed 42, notebook 03) | 4,17% | 2,5% a 6,9% | 0,83% | **17,8%** (679 SKUs) | 37,2% |
| Item2Vec v1 (média de 10 seeds) | 4,40% | de 3,9% a 5,1% entre seeds | — | — | — |

**Leitura:**
1. **Com os parâmetros da v2, o FP-Growth não gera nenhuma regra de SKU nesta base.** O par de SKUs mais frequente aparece em 6 cestas. Dentro de cada período, nenhum par chega ao mínimo que o `min_support` 0,01 exige (10 ocorrências na manhã, 4 na tarde). A recomendação cai inteira nas regras de categoria e na popularidade, e acerta o mesmo que os mais vendidos.
2. **Os números do Lucas vêm destes mesmos dados, só que em UTC.** No fuso UTC, o treino dele fica com tarde 1.164 / noite 121 / manhã 9, que é o que o log dele mostra. No horário de Brasília, o treino é 73% manhã e o teste é 94% tarde. Por isso a segmentação por período treina num horário e testa em outro.
3. **FP-Growth calibrado × Item2Vec v1 é empate estatístico:** 20 × 14 acertos, com 7 × 1 nas cestas discordantes (McNemar p = 0,07). O FP-Growth acerta um pouco mais. O Item2Vec cobre mais o catálogo (17,8% × 11,6%) e responde a mais carrinhos (85% × 65%). Na validação dentro do treino, a ordem se inverte: Item2Vec 10,8% × FP-Growth 3,6%.
4. **Achado principal: o acerto vem das cestas repetidas.** 35% das cestas de treino são cópias exatas de outra cesta (438 cestas, 161 conjuntos). Das 336 cestas de teste, 20 repetem uma cesta do treino, e os 20 acertos do FP-Growth calibrado são exatamente essas 20. O Item2Vec acerta 13 delas. Nas **316 cestas novas**, o FP-Growth acerta 0, o Item2Vec 1 e os mais vendidos 4. Hoje nenhum dos dois modelos generaliza para uma cesta nova: o Hit@5 mede quanto cada um **lembra** das cestas repetidas.
5. **O FP-Growth com piso 2 é, na prática, uma memória de cestas.** 14 dos 20 acertos vêm de regras vistas em só 2 cestas de treino, com pares improváveis como café Nespresso → biscoito Marilan (lift 623). Com piso 3, o acerto cai para 2,08%.
6. **O que mais pesou na mudança da v2 para o calibrado** (seção 8):
   * o piso de ocorrências levou de 1,49% para 4,76%;
   * tirar a segmentação não muda o acerto (4,46%), mas dobra a cobertura (4,2% → 8,7%);
   * ordenar por confiança acrescenta 4 acertos (5,65%).

### Ficha comparativa (base do one page)
| | FP-Growth (Modelo A · Lucas) | Item2Vec (Modelo B · Zaiden) |
|---|---|---|
| **Como funciona** | Conta quais SKUs aparecem juntos e gera regras "quem leva X e Y leva Z", com suporte, confiança e lift. | Aprende um vetor por SKU (Word2Vec, com as cestas como frases). Recomenda os SKUs mais próximos da média dos vetores do carrinho. |
| **Entradas** | `id_pedido` (cesta), `sku`, `id_categoria` (camada 2), hora → período do dia (v2), `freq_venda` (fallback) | `id_pedido` (cesta), `sku`. Subcategoria só no fallback; data só no split. |
| **Tipo de dado em que vai melhor** | Cestas densas, com pares que se repetem muitas vezes (o mínimo de ocorrências precisa ser alcançável) e catálogo pequeno. | Catálogo grande e esparso, cauda longa: aproxima SKUs que nunca apareceram juntos, mas têm os mesmos vizinhos. |
| **Saída** | Regras legíveis e auditáveis | Vetores e similaridade; menos explicável |
| **Hit@5 no mesmo hold-out** | v2: 1,49% · calibrado: 5,95% (IC 3,9% a 9,0%) | 4,17% (IC 2,5% a 6,9%) · média de 10 seeds: 4,40% |
| **Cobertura do catálogo** | v2: 1,0% · calibrado: 11,6% | 17,8% |
| **Cestas novas (316)** | 0 acertos | 1 acerto |
| **Pontos fortes** | Determinístico, explicável, acerta todas as cestas repetidas | Mais cobertura, responde a mais carrinhos, sugere itens da mesma missão de compra |
| **Limites** | Muito sensível ao piso (2 → 3 derruba de 5,95% para 2,08%); 161 mil regras, a maioria vista em 2 cestas | Varia com a seed (3,9% a 5,1%); tende a sugerir **substitutos** (ex.: 5 areias sanitárias para um carrinho com areia sanitária) |
| **Infra** | `mlxtend` ou `pyspark.ml.fpm`; regras numa tabela | `gensim` no driver; vetores numa tabela Delta |

### Próximos passos
1. **Evolução do Item2Vec (v2):** ajustar parâmetros na validação e medir o Hit@5 **separado entre cestas repetidas e novas**, porque melhorar só nas repetidas é decorar melhor, não generalizar.
2. **Sugestões para a v2 do Lucas:**
   * trocar o `min_support` fixo por um piso de ocorrências;
   * definir o fuso da sessão;
   * só segmentar por período se houver volume nos dois lados do split;
   * ordenar os consequentes por confiança;
   * avaliar todas as cestas de teste, inclusive as de períodos sem regras.
3. **Levar à squad a questão das cestas repetidas:** é recompra legítima ou artefato do gerador de dados? Se for artefato, deduplicar antes de treinar e avaliar.